# Step 4 — Create momentum and the prediction target

For each trading date, we calculate Apple's return over the **previous** five trading days. We then record whether Apple outperformed SPY over the **following** five trading days.

The future returns are used only to create the answer we will eventually predict; they are not model inputs.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "aapl_spy_2023-01-01_to_2025-01-01.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "aapl_momentum_target.csv"

raw = pd.read_csv(RAW_PATH, header=[0, 1], index_col=0, parse_dates=True)
adjusted_close = raw["Adj Close"][["AAPL", "SPY"]].copy()
adjusted_close.head()

Ticker,AAPL,SPY
Date,,
2023-01-03,122.982719,364.134003
2023-01-04,124.251183,366.945129
2023-01-05,122.933533,362.757050
2023-01-06,127.456787,371.075806
2023-01-09,127.977943,370.865448


## Build the research columns

`pct_change(5)` looks backward five rows, so it creates five-trading-day momentum.

`shift(-5)` brings the price from five rows ahead onto the current row, allowing us to calculate the two future returns.

In [2]:
research = adjusted_close.rename(columns={
    "AAPL": "aapl_adjusted_close",
    "SPY": "spy_adjusted_close",
})

research["aapl_momentum_5d"] = research["aapl_adjusted_close"].pct_change(5)
research["aapl_future_return_5d"] = (
    research["aapl_adjusted_close"].shift(-5)
    / research["aapl_adjusted_close"]
    - 1
)
research["spy_future_return_5d"] = (
    research["spy_adjusted_close"].shift(-5)
    / research["spy_adjusted_close"]
    - 1
)

# Keep the target missing when a future return is unavailable.
has_future = research[["aapl_future_return_5d", "spy_future_return_5d"]].notna().all(axis=1)
research["outperformed"] = pd.Series(pd.NA, index=research.index, dtype="Int64")
research.loc[has_future, "outperformed"] = (
    research.loc[has_future, "aapl_future_return_5d"]
    > research.loc[has_future, "spy_future_return_5d"]
).astype(int)

research.head(8)

Ticker,aapl_adjusted_close,spy_adjusted_close,aapl_momentum_5d,aapl_future_return_5d,spy_future_return_5d,outperformed
Date,,,,,,
2023-01-03,122.982719,364.134003,NaN,0.045254,0.025629,1
2023-01-04,124.251183,366.945129,NaN,0.056426,0.030644,1
2023-01-05,122.933533,362.757050,NaN,0.067109,0.046339,1
2023-01-06,127.456787,371.075806,NaN,0.039654,0.026850,1
2023-01-09,127.977943,370.865448,NaN,0.044487,0.025551,1
2023-01-10,128.548233,373.466339,0.045254,0.034269,0.002330,1
2023-01-11,131.262222,378.189850,0.056426,0.013334,-0.017395,1
2023-01-12,131.183533,379.566772,0.067109,0.033431,-0.002721,1


## Manually verify one row

We choose the sixth usable research row and calculate every value directly from the relevant prices. The assertions will fail if our vectorized formulas point in the wrong direction or use the wrong dates.

In [3]:
check_position = 10
check_date = research.index[check_position]
past_date = research.index[check_position - 5]
future_date = research.index[check_position + 5]

manual_momentum = (
    research.iloc[check_position]["aapl_adjusted_close"]
    / research.iloc[check_position - 5]["aapl_adjusted_close"]
    - 1
)
manual_aapl_future = (
    research.iloc[check_position + 5]["aapl_adjusted_close"]
    / research.iloc[check_position]["aapl_adjusted_close"]
    - 1
)
manual_spy_future = (
    research.iloc[check_position + 5]["spy_adjusted_close"]
    / research.iloc[check_position]["spy_adjusted_close"]
    - 1
)
manual_target = int(manual_aapl_future > manual_spy_future)

assert abs(manual_momentum - research.loc[check_date, "aapl_momentum_5d"]) < 1e-12
assert abs(manual_aapl_future - research.loc[check_date, "aapl_future_return_5d"]) < 1e-12
assert abs(manual_spy_future - research.loc[check_date, "spy_future_return_5d"]) < 1e-12
assert manual_target == research.loc[check_date, "outperformed"]

pd.Series({
    "Past date": past_date.date(),
    "Prediction date": check_date.date(),
    "Future date": future_date.date(),
    "AAPL momentum": manual_momentum,
    "AAPL future return": manual_aapl_future,
    "SPY future return": manual_spy_future,
    "Outperformed": manual_target,
})

Past date             2023-01-10
Prediction date       2023-01-18
Future date           2023-01-25
AAPL momentum           0.034269
AAPL future return      0.049183
SPY future return       0.022632
Outperformed                   1
dtype: object

## Create the usable research table

The first five dates lack prior history and the final five dates lack future outcomes. We remove only those incomplete rows from the processed table. The raw file remains unchanged.

In [4]:
modeling_data = research.dropna().copy()
modeling_data.index.name = "date"
modeling_data.to_csv(PROCESSED_PATH)

print(f"Saved {len(modeling_data):,} complete rows to {PROCESSED_PATH}")
print("First usable date:", modeling_data.index.min().date())
print("Last usable date:", modeling_data.index.max().date())
display(modeling_data.head())

Saved 492 complete rows to /Users/keishakalra/Desktop/Financial_App/data/processed/aapl_momentum_target.csv
First usable date: 2023-01-10
Last usable date: 2024-12-23


Ticker,aapl_adjusted_close,spy_adjusted_close,aapl_momentum_5d,aapl_future_return_5d,spy_future_return_5d,outperformed
date,,,,,,
2023-01-10,128.548233,373.466339,0.045254,0.034269,0.002330,1
2023-01-11,131.262222,378.189850,0.056426,0.013334,-0.017395,1
2023-01-12,131.183533,379.566772,0.067109,0.033431,-0.002721,1
2023-01-13,132.510986,381.039276,0.039654,0.047121,0.005345,1
2023-01-17,133.671341,380.341309,0.044487,0.048477,0.006109,1


In [5]:
summary = pd.Series({
    "Complete observations": len(modeling_data),
    "AAPL outperformed": int(modeling_data["outperformed"].sum()),
    "AAPL did not outperform": int((modeling_data["outperformed"] == 0).sum()),
    "Outperformance rate": modeling_data["outperformed"].mean(),
    "Average 5-day momentum": modeling_data["aapl_momentum_5d"].mean(),
})
summary

Complete observations      492.000000
AAPL outperformed          265.000000
AAPL did not outperform    227.000000
Outperformance rate          0.538618
Average 5-day momentum       0.007568
dtype: float64